# **NBA Data Gathering**

In [ ]:
!pip install bing-image-downloader

In [4]:
import requests
import pandas as pd
import seaborn as sns
import os
import numpy as np
import matplotlib.pyplot as plt
from bing_image_downloader import downloader

In [13]:
# Link for gathering all NBA players statistics all time
player_index = "https://stats.nba.com/stats/playerindex?College=&Country=&DraftPick=&DraftRound=&DraftYear=&Height=&Historical=1&LeagueID=00&Season=2024-25&SeasonType=Regular%20Season&TeamID=0&Weight="

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Origin": "https://stats.nba.com",
    "Referer": "https://stats.nba.com/stats/playerindex"
}

response = requests.get(player_index, headers=headers)

# Checking if scrape attempt worked successfully and storing the data
if response.status_code == 200:

    data = response.json()
    result_sets = data['resultSets'][0]
    headers = result_sets['headers']
    row_set = result_sets['rowSet']

else:

    print("Data scrapping failed.")


In [14]:
# Storing the scraped data in a pandas dataframe and creating a secondary dataframe of only current NBA players
df = pd.DataFrame(row_set, columns=headers)
df_current = df[df['ROSTER_STATUS'] == 1.0]
df_current.reset_index(drop=True, inplace=True)
df_current = df_current.drop(columns=['IS_DEFUNCT', 'STATS_TIMEFRAME', 'ROSTER_STATUS', 'PLAYER_SLUG', 'TEAM_SLUG'])
df_current['DRAFT_NUMBER'] = df_current['DRAFT_NUMBER'].fillna('Undrafted')
df_current['DRAFT_ROUND'] = df_current['DRAFT_ROUND'].fillna('Undrafted')

In [ ]:
# Creating a dict for labeling teams for future training
team_labels = {}
count = 0
for team in df_current['TEAM_NAME'].unique():
    team_labels[team] = count
    count += 1

In [ ]:
# Creating directory to store images for future scraping
os.makedirs("BingImages", exist_ok=True)

# Creating a unique directory for each team
for team in df_current['TEAM_NAME'].unique():
    os.makedirs(f"BingImages/{team}", exist_ok=True)

# **Image Scraping**

In [ ]:
# Scraping 10 images for each player listed in the dataframe and storing them in their respective teams directory
for idx, row in df_current.iterrows():
    search = f"{row['PLAYER_FIRST_NAME']} {row['PLAYER_LAST_NAME']} {row['TEAM_NAME']}"
    file_path = f"BingImages/{row['TEAM_NAME']}/"
    print(f"Currently on player {idx + 1} of {len(df_current)}")
    downloader.download(search, limit=10, output_dir=file_path, adult_filter_off=True, force_replace=False, timeout=60)

In [ ]:
# Removing files that do not contain the proper extension for training
valid_extensions = ['.jpg', '.jpeg', '.png', '.JPG']
for team in df_current['TEAM_NAME'].unique():
  folderpath = f"/content/BingImages/{team}/"
  for directory in os.listdir(folderpath):
    for filename in os.listdir(folderpath + directory):
      name, extension = os.path.splitext(filename)
      if extension not in valid_extensions:
        os.remove(folderpath + directory + "/" + filename)
        print(f'Removed {filename}')

In [ ]:
!zip BingImages.zip -r BingImages

# **Renaming Useable Images**

In [ ]:
!unzip UseableImages.zip

In [ ]:
# Renaming images to properly label which team the image belongs to
for directory in os.listdir("UseableImages"):
  count = 0
  for filename in os.listdir(f"UseableImages/{directory}"):
    os.rename(f"UseableImages/{directory}/{filename}", f"UseableImages/{directory}/{directory}_{count}.jpg")
    count += 1

In [ ]:
!zip UseableImageClean.zip -r UseableImages